# Real Estate Housing Price Prediction
# Parts 1 to 4 - Complete Project
### Uses: numpy, pandas, matplotlib only (no sklearn for core algorithms)

In [ ]:
# Basic imports - only allowed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set seed so results are same every time
np.random.seed(42)

print('All imports done!')

## Step 1: Generate Dataset

In [ ]:
# Generate the real estate dataset as given in project
np.random.seed(42)
n_samples = 10000

# Generate features with some correlation between them
area = np.random.normal(2000, 500, n_samples)
bedrooms = np.random.poisson(3, n_samples) + 1
bathrooms = bedrooms * 0.8 + np.random.normal(0, 0.5, n_samples)
age = np.random.exponential(15, n_samples)
distance_city = np.random.gamma(2, 3, n_samples)
crime_rate = np.random.exponential(5, n_samples)
school_rating = np.random.beta(2, 1, n_samples) * 9 + 1
garage = np.random.binomial(3, 0.6, n_samples)
basement = area * 0.3 + np.random.normal(0, 200, n_samples)

# Price formula with non-linear term and interaction term
price = (150 * area +
         10000 * bedrooms +
         8000 * bathrooms -
         300 * age -
         2000 * distance_city -
         1000 * crime_rate +
         5000 * school_rating +
         3000 * garage +
         50 * basement +
         0.01 * area**2 -
         100 * age * distance_city +
         np.random.normal(0, 20000, n_samples))

# Put all data into a dataframe
df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age': age,
    'distance_city': distance_city,
    'crime_rate': crime_rate,
    'school_rating': school_rating,
    'garage': garage,
    'basement': basement,
    'price': price
})

print('Dataset shape:', df.shape)
print('\\nFirst 5 rows:')
print(df.head())

In [ ]:
# Add 5% missing values randomly in all columns except price
np.random.seed(42)
missing_rate = 0.05

for col in df.columns:
    if col != 'price':
        missing_idx = np.random.choice(df.index, size=int(missing_rate * len(df)), replace=False)
        df.loc[missing_idx, col] = np.nan

# Add 2% outliers in price
outlier_idx = np.random.choice(df.index, size=int(0.02 * len(df)), replace=False)
df.loc[outlier_idx, 'price'] = df['price'].mean() + np.random.choice([-1, 1], size=len(outlier_idx)) * df['price'].std() * 5

print('Missing values per column:')
print(df.isnull().sum())
print('\\nDataset ready with missing values and outliers added!')

## Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print('Basic statistics:')
print(df.describe().round(2))

In [ ]:
# Plot distributions of all features
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Distributions plotted!')

---
# PART 1: Simple Linear Regression (25 points)

## 1.1 Handle Missing Values - Two Methods

In [ ]:
# Method 1: Fill missing values with mean
df_mean_imputed = df.copy()
for col in df_mean_imputed.columns:
    if col != 'price':
        col_mean = df_mean_imputed[col].mean()
        df_mean_imputed[col].fillna(col_mean, inplace=True)

print('Method 1: Mean imputation done')
print('Missing after mean imputation:', df_mean_imputed.isnull().sum().sum())

In [ ]:
# Method 2: Fill missing values with median
df_median_imputed = df.copy()
for col in df_median_imputed.columns:
    if col != 'price':
        col_median = df_median_imputed[col].median()
        df_median_imputed[col].fillna(col_median, inplace=True)

print('Method 2: Median imputation done')
print('Missing after median imputation:', df_median_imputed.isnull().sum().sum())

## 1.2 Remove Outliers using IQR Method

In [ ]:
def remove_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    cleaned = dataframe[(dataframe[column] >= lower) & (dataframe[column] <= upper)]
    return cleaned

# Apply on mean imputed data
df_clean = remove_outliers_iqr(df_mean_imputed.dropna(), 'price')
df_with_outliers = df_mean_imputed.dropna().copy()

print(f'Rows before removing outliers: {len(df_with_outliers)}')
print(f'Rows after removing outliers: {len(df_clean)}')
print(f'Outliers removed: {len(df_with_outliers) - len(df_clean)}')

## 1.3 SimpleLinearRegression Class (from scratch)

In [ ]:
class SimpleLinearRegression:
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.slope = 0
        self.intercept = 0
        self.cost_history = []

    def compute_cost(self, X, y):
        n = len(y)
        predictions = self.slope * X + self.intercept
        errors = predictions - y
        cost = np.sum(errors ** 2) / (2 * n)
        return cost

    def fit(self, X, y):
        n = len(y)
        self.slope = 0.0
        self.intercept = 0.0
        self.cost_history = []
        prev_cost = float('inf')

        for i in range(self.max_iterations):
            predictions = self.slope * X + self.intercept
            errors = predictions - y

            grad_slope = np.sum(errors * X) / n
            grad_intercept = np.sum(errors) / n

            current_lr = self.learning_rate / (1 + 0.001 * i)

            self.slope -= current_lr * grad_slope
            self.intercept -= current_lr * grad_intercept

            current_cost = self.compute_cost(X, y)
            self.cost_history.append(current_cost)

            if abs(prev_cost - current_cost) < self.tolerance:
                print(f'Converged at iteration {i}')
                break
            prev_cost = current_cost

        return self

    def predict(self, X):
        return self.slope * X + self.intercept

    def r_squared(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print('SimpleLinearRegression class created!')

In [ ]:
# Prepare data: predict price using area only
X_simple = df_clean['area'].values
y_simple = df_clean['price'].values

# Normalize X
X_mean = X_simple.mean()
X_std = X_simple.std()
X_norm = (X_simple - X_mean) / X_std

y_mean = y_simple.mean()
y_std = y_simple.std()
y_norm = (y_simple - y_mean) / y_std

# Train the model
slr = SimpleLinearRegression(learning_rate=0.1, max_iterations=2000, tolerance=1e-8)
slr.fit(X_norm, y_norm)

print(f'Slope: {slr.slope:.4f}')
print(f'Intercept: {slr.intercept:.4f}')
print(f'R2 Score: {slr.r_squared(X_norm, y_norm):.4f}')

In [ ]:
# Plot cost function
plt.figure(figsize=(10, 4))
plt.plot(slr.cost_history, color='crimson', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE)')
plt.title('Cost Function Convergence During Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Cost converged nicely!')

In [ ]:
# Plot regression line
y_pred_norm = slr.predict(X_norm)
y_pred_orig = y_pred_norm * y_std + y_mean

residuals = y_simple - y_pred_orig
std_error = np.std(residuals)
n = len(y_simple)
ci = 1.96 * std_error / np.sqrt(n)

sort_idx = np.argsort(X_simple)
X_sorted = X_simple[sort_idx]
y_pred_sorted = y_pred_orig[sort_idx]

plt.figure(figsize=(12, 6))
plt.scatter(X_simple, y_simple, alpha=0.2, color='steelblue', s=5, label='Actual data')
plt.plot(X_sorted, y_pred_sorted, color='red', linewidth=2, label='Regression line')
plt.fill_between(X_sorted, y_pred_sorted - ci, y_pred_sorted + ci,
                 alpha=0.2, color='orange', label='95% Confidence Band')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price ($)')
plt.title('Simple Linear Regression: Area vs Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_pred_orig, residuals, alpha=0.3, s=5, color='steelblue')
axes[0].axhline(y=0, color='red', linewidth=2)
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3)

residuals_sorted = np.sort(residuals)
n = len(residuals_sorted)
theoretical_q = np.array([np.percentile(np.random.normal(0, 1, 1000), 100 * i / n) for i in range(1, n + 1)])
axes[2].scatter(theoretical_q, residuals_sorted, alpha=0.3, s=5, color='steelblue')
axes[2].set_xlabel('Theoretical Quantiles')
axes[2].set_ylabel('Sample Quantiles')
axes[2].set_title('Q-Q Plot')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()